.pth to .tflite Converter ----
General pipeline: PyTorch .pth -> ONNX -> TensorFlow Model -> .tflite

In [1]:
!pip install onnx2tf tensorflow

In [4]:
import os
import torch
from pathlib import Path
import torchvision.models as models
import onnx
import tensorflow as tf

In [5]:
# add model to content folder and adjust model name appropriately
model_path = Path(os.getcwd()) / "food101_efficientnetv2s.pth"

if not model_path.exists():
    raise FileNotFoundError(f"Model not found at: {model_path}")

print(f"Model found at: {model_path}")

Model found at: /content/food101_efficientnetv2s.pth


In [6]:

# Recreate the same architecture used during training
model = models.efficientnet_v2_s(weights=None)

# Adjust the classifier head to match your 101 food classes
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 101)

# Load the saved weights into the model
state_dict = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
model.load_state_dict(state_dict)

model.eval()



EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [7]:
# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy_input,
    "food101_efficientnetv2s.onnx",
    opset_version=13,
    input_names=["input"],
    output_names=["output"],
)

print ("successfully converted to ONNX")

W0413 15:33:34.602000 6905 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0413 15:33:35.728000 6905 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 15:33:35.730000 6905 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -

[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

Applied 220 of general pattern rewrite rules.
successfully converted to ONNX


In [8]:
# Convert ONNX to TF SavedModel to TFLite (onnx2tf does both in one step)
!onnx2tf -i food101_efficientnetv2s.onnx -o food101_tf_model -osd -cotof




Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 35             │ 35               │
│ Constant   │ 344            │ 344              │
│ Conv       │ 170            │ 170              │
│ Gemm       │ 1              │ 1                │
│ Mul        │ 132            │ 132              │
│ ReduceMean │ 31             │ 31               │
│ Reshape    │ 1              │ 1                │
│ Sigmoid    │ 132            │ 132              │
│ Model Size │ 78.4MiB        │ 77.3MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 

In [12]:

# download as tflite model
from google.colab import files

files.download(Path(os.getcwd()) / 'food101_tf_model/food101_efficientnetv2s_float32.tflite')
# /content/food101_tf_model/food101_efficientnetv2s_float32.tflite
# /content/food101_tf_model/food101_efficientnetv2s.tflite

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>